#### Collocation Matrix

In [ ]:
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys

if "google.colab" in sys.modules:
    required_solvers = set()

    if importlib.util.find_spec("pyomo") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pyomo"])

    apt_packages = []
    if "glpk" in required_solvers and shutil.which("glpsol") is None:
        apt_packages.append("glpk-utils")
    if "cbc" in required_solvers and shutil.which("cbc") is None:
        apt_packages.append("coinor-cbc")

    if apt_packages:
        subprocess.check_call(["apt-get", "update"])
        subprocess.check_call(["apt-get", "install", "-y", "-qq", *apt_packages])

    if "ipopt" in required_solvers and shutil.which("ipopt") is None:
        if importlib.util.find_spec("idaes") is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "idaes-pse"])
        idaes_bin = Path.home() / ".idaes" / "bin"
        subprocess.check_call(["idaes", "get-extensions"])
        os.environ["PATH"] = f"{idaes_bin}:{os.environ['PATH']}"

    missing_solvers = []
    if "glpk" in required_solvers and shutil.which("glpsol") is None:
        missing_solvers.append("glpk")
    if "cbc" in required_solvers and shutil.which("cbc") is None:
        missing_solvers.append("cbc")
    if "ipopt" in required_solvers and shutil.which("ipopt") is None:
        missing_solvers.append("ipopt")

    if missing_solvers:
        raise RuntimeError(
            "Missing solver executables after Colab setup: "
            + ", ".join(sorted(missing_solvers))
        )


In [1]:
import numpy

cp = [0, 0.155051, 0.644949, 1]

a = []

print('[')
for i in range(len(cp)):
    ptmp = []
    tmp = 0
    for j in range(len(cp)):
        if j != i:
            row = []
            row.insert(0,1/(cp[i]-cp[j]))
            row.insert(1,-cp[j]/(cp[i]-cp[j]))
            ptmp.insert(tmp,row)
            tmp += 1
    p=[1]
    for j in range(len(cp)-1):
        p = numpy.convolve(p,ptmp[j])
    pder = numpy.polyder(p,1)
    arow = []
    for j in range(len(cp)):
        arow.append(numpy.polyval(pder,cp[j]))
    a.append(arow)
    print(str(arow)+',')
print(']')

[
[np.float64(-9.000001008080126), np.float64(-4.139388773624379), np.float64(1.7393879671602779), np.float64(-3.0000002520200333)],
[np.float64(10.048810106494384), np.float64(3.2247461916839306), np.float64(-3.567840077120941), np.float64(5.531972415060627)],
[np.float64(-1.382142403745367), np.float64(1.1678398419022438), np.float64(0.7752546483828548), np.float64(-7.53197233105394)],
[np.float64(0.33333330533110994), np.float64(-0.25319725996179565), np.float64(1.0531974615778044), np.float64(5.000000168013341)],
]
